# 07 — Simulation analysis

This notebook runs a ROI-space null-model simulation validating the permutation inference.

The simulation builds null studies in which there is, by construction, *no* true
lesion–symptom relationship: lesions are sampled (random ROIs or resampled real lesions),
each lesion is given a random symptom value, and a lesion-network map is computed by
projecting through the functional connectome `C`. Because the symptom values are random,
any apparent association is a false positive.

We assess significance with the two-sided
   permutation p-value, `p = (#{|null| ≥ |obs|} + 1) / (n_perms + 1)`, thresholded at `p < 0.05`. If the procedure is valid the proportion of "significant" ROIs (the empirical false-positive rate, FPR) should sit at the nominal 5 % across studies.

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.decomposition import PCA
from joblib import Parallel, delayed

## Parameters

The simulation is symptom-agnostic (random symptom values), so the null model is identical
across cognitive domains and does not depend on any particular LNM design — it needs only the
group functional connectome (used to define `C`, degree, and PC1). Random seeds and simulation
counts are fixed (`N_SIM = 10000`, `n_perms = 1000`, `seed = range(N_SIM)`).

In [ ]:
N_SIM            = 10000          # number of null studies per arm
N_PERMS          = 1000          # permutations per null study
N_LESIONS        = 50            # lesions per null study
LESION_SIZE_RANGE = (1, 51)      # random-lesion size range (ROIs), upper-exclusive
SEEDS            = range(N_SIM)  # reproducible seeds

print(f"N_SIM={N_SIM}  N_PERMS={N_PERMS}  N_LESIONS={N_LESIONS}  size={LESION_SIZE_RANGE}")

## Connectome, degree, and PC1

In [ ]:
def load_connectome():
    """Group FC matrix C (n_roi x n_roi). Prefers the parcellated TSV; falls back to a .mat file."""
    # Parcellated group connectivity matrix (Schaefer2018 + Tian2020, 1054 ROIs).
    tsv_candidates = sorted((DATA_DIR / "connectomes").glob(
        "*parcels1054*desc-groupconnectivity_connmatrix.tsv"))
    for tsv in tsv_candidates:
        try:
            C = pd.read_csv(tsv, sep="\t", index_col=0).values.astype(float)
            print(f"Connectome (TSV): {tsv.name}  shape={C.shape}")
            return C
        except Exception as e:
            print(f"  could not read {tsv.name}: {e}")
    # Fallback: scipy .mat holding the group FC matrix.
    from scipy.io import loadmat
    for mat in [DATA_DIR / "GSP1000_FC_Schaefer1000Melbourne54.mat",
                PROJECT_ROOT.parent / "data" / "GSP1000_FC_Schaefer1000Melbourne54.mat"]:
        if mat.exists():
            C = loadmat(mat)["FC"]
            print(f"Connectome (.mat): {mat}  shape={C.shape}")
            return C
    return None

C = load_connectome()
if C is None:
    print("Connectome not found — cannot run simulation.")
    pc1 = degree = None
    n_rois = None
else:
    n_rois = C.shape[0]
    pc1 = PCA(n_components=1).fit_transform(zscore(C, axis=0)).flatten()
    degree = np.sum(C, axis=0)
    print(f"n_rois = {n_rois}  | degree range [{degree.min():.1f}, {degree.max():.1f}]")

## Real-lesion overlap matrix

There are two arms: **synthetic** (random ROI) lesions and **real** lesions resampled
from an actual lesion-overlap matrix `M_real` (subjects × ROIs, binary). The CSV
`lesion_overlap_schaefer1000_roi_data.csv` is looked up under `output/.../`.

In [ ]:
def load_real_lesions():
    """Binary lesion-overlap matrix M_real (n_subjects x n_roi), or None if unavailable."""
    candidates = [
        # Preferred locations:
        PROJECT_ROOT / "output" / "matters_arising" / "lesion_overlap_schaefer1000_roi_data.csv",
        PROJECT_ROOT / "data" / "processed" / "lesion_overlap_schaefer1000_roi_data.csv",
        # Additional fallback location:
        PROJECT_ROOT.parent / "outputs" / "matters_arising" / "lesion_overlap_schaefer1000_roi_data.csv",
    ]
    for c in candidates:
        if c.exists():
            M = pd.read_csv(c, index_col=0).values
            print(f"Real lesions: {M.shape}  <- {c}")
            return M
    print("Real-lesion overlap matrix not found — the real-lesion arm will be SKIPPED.")
    print("  (looked in: " + "; ".join(str(c) for c in candidates) + ")")
    return None

M_real = load_real_lesions()

## Simulation function

One call = one **null study**. Identical logic to the reference:
random/resampled lesions → connectome projection → random symptom values → observed map →
1000 sign permutations → two-sided permutation p-value and the signed
`−log10 p` "permutation-corrected" map. Returns the per-study FPR, the four R² values
(raw/corrected × degree/PC1), and the per-ROI significance vector (for the spatial FPR).

In [ ]:
def run_simulation(C, pc1_map, degree_map, seed, n_lesions=N_LESIONS,
                   lesion_size_range=LESION_SIZE_RANGE, n_perms=N_PERMS, M=None):
    """One null study (no true lesion-symptom link).

    Inference is the STANDARD two-sided permutation p-value with the +1 correction
    (Phipson & Smyth, 2010): p = (#{|null| >= |obs|} + 1) / (n_perms + 1), thresholded at
    p < 0.05. The former null mean/SD-derived z-statistic (|Z| > 1.96) is no longer used
    anywhere. The 'permutation-corrected' map used in the R^2-with-degree/PC1
    panels is the signed permutation evidence, sign(obs - null_mean) * -log10(p) -- a purely
    permutation-based quantity, not a z-score.
    """
    n_rois = C.shape[0]
    rng = np.random.RandomState(seed)
    if M is not None:
        M_sim = M[rng.choice(M.shape[0], n_lesions, replace=True), :]
    else:
        M_sim = np.zeros((n_lesions, n_rois))
        for i in range(n_lesions):
            size = rng.randint(lesion_size_range[0], lesion_size_range[1])
            M_sim[i, rng.choice(n_rois, size, replace=False)] = 1

    L_conn = M_sim @ C
    sv = rng.randn(n_lesions)                       # random symptom values
    obs = sv @ L_conn                               # observed lesion-network map

    perms = np.array([rng.permutation(sv) for _ in range(n_perms)])
    null = perms @ L_conn                           # permutation null (n_perms x n_rois)

    # --- standard permutation p-value (two-sided) -> significance / FPR ---
    count = (np.abs(null) >= np.abs(obs)).sum(axis=0)
    p_perm = (count + 1) / (n_perms + 1)
    sig_perm = (p_perm < 0.05)

    # --- permutation-corrected map (replaces the z-map): signed -log10(p) ---
    corr_map = np.sign(obs - null.mean(axis=0)) * (-np.log10(p_perm))

    r2_raw_deg = np.corrcoef(obs, degree_map)[0, 1] ** 2
    r2_corr_deg = np.corrcoef(corr_map, degree_map)[0, 1] ** 2
    r2_raw_pc1 = np.corrcoef(obs, pc1_map)[0, 1] ** 2
    r2_corr_pc1 = np.corrcoef(corr_map, pc1_map)[0, 1] ** 2

    return {
        "FPR_perm": sig_perm.mean(),         # proportion of significant ROIs (perm p<.05)
        "R2_Raw_Degree": r2_raw_deg, "R2_Corr_Degree": r2_corr_deg,
        "R2_Raw_PC1": r2_raw_pc1, "R2_Corr_PC1": r2_corr_pc1,
        "_sig_perm": sig_perm.astype(np.int8),   # per-ROI significance (for spatial FPR)
    }

## Run the null studies (synthetic + real lesions)

10,000 null studies per arm, parallelised over seeds.

In [ ]:
CACHE_DIR = PROJECT_ROOT / "output" / "sim_cache"

def run_arm(M, tag):
    _key = tag.replace(" ", "_")
    _cdf  = CACHE_DIR / f"df_{_key}_nsim{N_SIM}_nperm{N_PERMS}.pkl"
    _croi = CACHE_DIR / f"roi_{_key}_nsim{N_SIM}_nperm{N_PERMS}.npy"
    if _cdf.exists() and _croi.exists():
        df = pd.read_pickle(_cdf); per_roi_fpr = np.load(_croi)
        print(f"[{tag}] loaded cached results ({len(df)} sims): {_cdf.name}")
    else:
        results = Parallel(n_jobs=-1, verbose=0)(
            delayed(run_simulation)(C, pc1, degree, seed, M=M) for seed in SEEDS
        )
        df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in results])
        df["FPR_perm_pct"] = df["FPR_perm"] * 100
        per_roi_fpr = np.vstack([r["_sig_perm"] for r in results]).mean(axis=0)  # per-ROI FPR
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        df.to_pickle(_cdf); np.save(_croi, per_roi_fpr)
        print(f"[{tag}] computed {len(df)} sims and cached -> {_cdf.name}")
    print(f"[{tag}] permutation-p FPR  = {df['FPR_perm_pct'].mean():.2f}%")
    print(f"[{tag}] per-ROI FPR mean   = {per_roi_fpr.mean()*100:.2f}%  | "
          f"corr(FPR, degree) = {np.corrcoef(per_roi_fpr, degree)[0,1]:+.3f}  | "
          f"corr(FPR, |PC1|) = {np.corrcoef(per_roi_fpr, np.abs(pc1))[0,1]:+.3f}")
    print(f"[{tag}] R^2 with degree: raw = {df['R2_Raw_Degree'].mean():.3f}  ->  "
          f"corrected = {df['R2_Corr_Degree'].mean():.3f}")
    print(f"[{tag}] R^2 with PC1:    raw = {df['R2_Raw_PC1'].mean():.3f}  ->  "
          f"corrected = {df['R2_Corr_PC1'].mean():.3f}")
    return df, per_roi_fpr

df_rand = fpr_roi_rand = None
df_real = fpr_roi_real = None

if C is None:
    print("Connectome not available — skipping simulation. (PALM not finished / inputs not found.)")
else:
    df_rand, fpr_roi_rand = run_arm(None, "synthetic lesions")
    if M_real is not None:
        df_real, fpr_roi_real = run_arm(M_real, "real lesions")
    else:
        print("[real lesions] skipped (no lesion-overlap matrix).")

## Figure — Simulation panel

Two rows: **a** = synthetic (random) lesions, **b** = real lesions.
Columns: (1) distribution of the per-study FPR with the nominal 5 % line, (2) R² with nodal
degree (raw vs permutation-corrected), (3) R² with connectome PC1 (raw vs corrected).

In [ ]:
def _plot_row(ax, df, rowlab, fontsize):
    mean_fpr = df["FPR_perm_pct"].mean()
    sns.histplot(df["FPR_perm_pct"], color="red", ax=ax[0], bins=50, alpha=0.6)
    ax[0].axvline(mean_fpr, color="red", lw=2, label=f"Observed mean ({mean_fpr:.1f}%)")
    ax[0].axvline(5.0, color="black", ls="--", lw=2, label="Target type I error (5.0%)")
    ax[0].set_xlabel("Proportion of significant ROIs (%)", fontsize=fontsize)
    ax[0].set_ylabel("Count", fontsize=fontsize)
    ax[0].tick_params(labelsize=fontsize); ax[0].legend(fontsize=fontsize - 6)

    sns.histplot(df["R2_Raw_Degree"], color="gray", label="No label permutation", ax=ax[1], alpha=0.5, bins=30)
    sns.histplot(df["R2_Corr_Degree"], color="blue", label="Label permutation", ax=ax[1], alpha=0.5, bins=30)
    ax[1].set_xlabel("$R^2$ with nodal degree", fontsize=fontsize)
    ax[1].set_ylabel("Count", fontsize=fontsize)
    ax[1].tick_params(labelsize=fontsize); ax[1].legend(fontsize=fontsize - 6)

    sns.histplot(df["R2_Raw_PC1"], color="gray", label="No label permutation", ax=ax[2], alpha=0.5, bins=30)
    sns.histplot(df["R2_Corr_PC1"], color="blue", label="Label permutation", ax=ax[2], alpha=0.5, bins=30)
    ax[2].set_xlabel("$R^2$ with connectome PC1", fontsize=fontsize)
    ax[2].set_ylabel("Count", fontsize=fontsize)
    ax[2].tick_params(labelsize=fontsize); ax[2].legend(fontsize=fontsize - 6)

    ax[0].text(-0.02, 1.06, rowlab, transform=ax[0].transAxes,
               fontsize=28, fontweight="bold", ha="right", va="bottom")

_arms = [(df_rand, "a")] + ([(df_real, "b")] if df_real is not None else [])
_arms = [(d, lab) for d, lab in _arms if d is not None]

if not _arms:
    print("No simulation results to plot — run the simulation cells first "
          "(connectome / PALM inputs not available yet).")
else:
    fontsize = 22
    fig, axes = plt.subplots(len(_arms), 3, figsize=(20, 5.5 * len(_arms)), squeeze=False)
    for r, (df, rowlab) in enumerate(_arms):
        _plot_row(axes[r], df, rowlab, fontsize)
    plt.tight_layout()
    _stem = "07_simulation_permutation"
    fig.savefig(FIG_DIR / f"{_stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{_stem}.svg", bbox_inches="tight")
    plt.show()

## Summary table

In [ ]:
if df_rand is None:
    print("No results yet — summary skipped. (PALM not finished / connectome not found.)")
else:
    rows, arms = [], [("synthetic lesions", df_rand, fpr_roi_rand)]
    if df_real is not None:
        arms.append(("real lesions", df_real, fpr_roi_real))
    for tag, df, fpr_roi in arms:
        rows.append({
            "arm": tag,
            "FPR_perm_pct": df["FPR_perm_pct"].mean(),
            "R2_raw_degree": df["R2_Raw_Degree"].mean(),
            "R2_corr_degree": df["R2_Corr_Degree"].mean(),
            "R2_raw_PC1": df["R2_Raw_PC1"].mean(),
            "R2_corr_PC1": df["R2_Corr_PC1"].mean(),
            "perROI_FPR_corr_degree": np.corrcoef(fpr_roi, degree)[0, 1],
        })
    summary = pd.DataFrame(rows).round(3)
    _csv = FIG_DIR / "07_simulation_summary.csv"
    summary.to_csv(_csv, index=False)
    print(summary.to_string(index=False)); print()